# Animated spectral measures

This notebook makes the GIFs of the spectral measures $\mu_t^{\mathbf{U}}$, $\mu_t^{\mathbf{V}}$ and $\mu_t^{\mathbf{Z}}$, with one setting per phenomenon of the paper. The code is in `src/animations.py`.

- The high-dimensional densities (Theorem 4) are always computed. They need only NumPy, SciPy and Matplotlib, and take a few seconds per frame.
- With `SIMULATE = True`, gradient descent is also run at the same times, and the GIF overlays its histograms on the theory curves. This needs PyTorch, and a GPU is recommended. The fronts at $\rho = 10^{-25}$ need up to about $9 \times 10^4$ steps.
- Results are saved in `data/` as `animation_<name>_theory.npz` and `animation_<name>_gd.npz`, so changing only the appearance redraws the GIF without recomputing. To force a new computation, pass `recompute=True`. An archive is recomputed automatically when a setting that affects it changes.
- The GIFs are written to `animations/gifs/<name>.gif`.

Conventions are those of Figures 3, 4 and 8. Only the continuous part of each measure is drawn: the atom at zero, and the atom at $2\rho$ of $\mathbf{V}\mathbf{V}^\top$, are left out of both the curves and the histograms. The dashed curve is $\kappa^*\phi_C$ for a single $\mathbf{Z}$ panel, and the $t \to \infty$ limit of each measure when several panels are shown.

In [ ]:
from pathlib import Path
import sys

from IPython.display import Image, display

ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "src" / "paths.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not find the repository root (the folder containing src/).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import animations as A

# True: also run gradient descent and overlay its histograms (needs PyTorch, GPU recommended).
SIMULATE = True

# True: also write the dark versions of GIFs
DARK_GIFS = True

print("GIFs are written to", A.GIF_DIR)

## Settings

Each setting shows one phenomenon of the paper.

1. `transport_underparameterized` ($\kappa = 0.55 < \kappa^* = 0.9$, $\rho = 0.1$, $t$ on a log scale). The spectrum of $\mathbf{Z}$ moves from zero to the part of $\kappa^*\phi_C$ above the rank-selection threshold $x_*$ (Theorem 4, Corollary 5, Figure 4).
2. `transport_overparameterized` ($\kappa = 0.75 > \kappa^* = 0.55$). The whole spectrum of $\mathbf{C}$ is learned.
3. `gram_matrices` ($\rho = 0.3$, overparameterized). $\mathbf{U}\mathbf{U}^\top$, $\mathbf{V}\mathbf{V}^\top$ and $\mathbf{Z}$ side by side, each with its $t \to \infty$ limit (dashed): the spectrum of $\mathbf{V}\mathbf{V}^\top$ is that of $\mathbf{U}\mathbf{U}^\top$ shifted by $2\rho$, and the singular values of $\mathbf{Z}$ are $\sqrt{\lambda(\lambda + 2\rho)}$ (Proposition 2).
4. `delayed_learning` (underparameterized, $\rho = 0.1$, $10^{-3}$ and $10^{-10}$ on the same panel, $t$ on a log scale). The same transport happens later when $\rho$ is smaller, after a waiting time of order $\log(1/\rho)$. A second row shows the alignment error and the loss of gradient descent, both divided by their initial value: the directions align on a time of order one for the three values of $\rho$, while the loss waits (silent alignment, Figure 2).
5. `spectral_front_underparameterized` ($\rho = 10^{-25}$, $s = t/\log(1/\rho)$ from 0.15 to 1.6). Nothing moves at first (silent phase). Then the singular values above the moving threshold $x_s = 1/(2s)$ are learned, and the front stops at $x_*$ when $s = 1/(2x_*)$, because of the rank constraint (Theorem 12, Figure 8).
6. `spectral_front_overparameterized` (same, overparameterized). The front goes down to the lower edge of the spectrum, where it stops, and the whole spectrum is learned.

In [ ]:
COMMON = dict(n_frames=72, fps=12, hold_last=18, dpi=220, d=400, reps=10, eta=1e-3)

UNDER = dict(kappa=0.55, kappa_star=0.90)
OVER = dict(kappa=0.75, kappa_star=0.55)

FRONT = dict(time_axis="s", s_range=(0.15, 1.6), smoothing=5e-3, n_quad=7500, panels=("Z",))
FRONT_COMMON = {**COMMON, "n_frames": 188}

SETTINGS = [
    A.GifSettings(
        name="transport_underparameterized",
        title=r"Underparameterized, $\rho = 0.1$",
        rho=0.1, time_axis="log", t_min=0.1, t_max=15.0, panels=("Z",),
        x_star_label="plot", y_headroom=1.25,
        **UNDER, **COMMON,
    ),
    A.GifSettings(
        name="transport_overparameterized",
        title=r"Overparameterized, $\rho = 0.1$",
        rho=0.1, time_axis="log", t_min=0.1, t_max=15.0, panels=("Z",),
        x_star_label="plot", y_headroom=1.25,
        **OVER, **COMMON,
    ),
    A.GifSettings(
        name="gram_matrices",
        title=r"Overparameterized, $\rho = 0.3$",
        rho=0.3, time_axis="log", t_min=0.05, t_max=10.0, panels=("U", "V", "Z"),
        **OVER, **COMMON,
    ),
    A.GifSettings(
        name="delayed_learning",
        title=r"Underparameterized",
        rhos=(0.1, 1e-3, 1e-10), time_axis="log", t_min=0.2, t_max=40.0, panels=("Z",),
        curves=("alignment", "loss"), x_star_label="plot", y_headroom=1.25,
        **UNDER, **COMMON,
    ),
    A.GifSettings(
        name="spectral_front_underparameterized",
        title=r"Underparameterized, $\rho = 10^{-25}$",
        rho=1e-25,
        **FRONT, **UNDER, **FRONT_COMMON,
    ),
    A.GifSettings(
        name="spectral_front_overparameterized",
        title=r"Overparameterized, $\rho = 10^{-25}$",
        rho=1e-25,
        **FRONT, **OVER, **FRONT_COMMON,
    ),
]

assert len({s.name for s in SETTINGS}) == len(SETTINGS), "setting names must be unique"
for s in SETTINGS:
    print(A.describe(s), f"({s.quad_nodes} quadrature nodes)" if s.rhos is None else "")

## Preview one frame

Draws a single frame of one setting, to check the appearance before writing the GIFs. The frames are computed and cached on the first call, so the GIF of the same setting afterwards only redraws.

In [ ]:
PREVIEW = "transport_underparameterized"
PREVIEW_FRAME = 40

setting = next(s for s in SETTINGS if s.name == PREVIEW)
display(Image(filename=str(A.preview(setting, PREVIEW_FRAME, simulate=SIMULATE))))

## Make the GIFs

Writes `animations/gifs/<name>.gif` for every setting, then shows them. On the first run this computes the theory (a few minutes per setting, longer for the three-panel one) and, if `SIMULATE`, the gradient-descent runs. Later runs reuse the cache.

In [ ]:
THEMES = ("light", "dark") if DARK_GIFS else ("light",)

paths = {}
for setting in SETTINGS:
    print(f"\n=== {setting.name} ===")
    for theme in THEMES:
        paths[setting.name, theme] = A.make_gif(setting, simulate=SIMULATE, theme=theme)

for (name, theme), path in paths.items():
    print(name, theme)
    display(Image(filename=str(path)))